In [3]:
import pandas as pd
import os

print("Pandas version:", pd.__version__)
print("Preprocessing started")

Pandas version: 3.0.6
Preprocessing started


In [4]:
logon = pd.read_csv("../data/raw/r1/r1/logon.csv")

logon["date"] = pd.to_datetime(
    logon["date"],
    format="%m/%d/%Y %H:%M:%S"
)

print("Logon shape:", logon.shape)
print(logon.head())

Logon shape: (849579, 5)
                         id                date          user       pc  \
0  {Y6O4-A7KC67IN-0899AOZK} 2010-01-04 00:10:37  DTAA/KEE0997  PC-1914   
1  {O5Y6-O7CJ02JC-6704RWBS} 2010-01-04 00:52:16  DTAA/KEE0997  PC-1914   
2  {D2D1-C6EB14QJ-2100RSZO} 2010-01-04 01:17:20  DTAA/KEE0997  PC-3363   
3  {H9W1-X0MC70BT-6065RPAT} 2010-01-04 01:28:34  DTAA/KEE0997  PC-3363   
4  {H3H4-S5AZ00AZ-9560IYHC} 2010-01-04 01:57:30  DTAA/BJM0992  PC-3058   

  activity  
0    Logon  
1   Logoff  
2    Logon  
3   Logoff  
4    Logon  


In [5]:
# Load Device dataset
device = pd.read_csv("../data/raw/r1/r1/device.csv")

# Convert date
device["date"] = pd.to_datetime(
    device["date"],
    format="%m/%d/%Y %H:%M:%S"
)

print("Device shape:", device.shape)
print(device.head())

Device shape: (65668, 5)
                         id                date          user       pc  \
0  {S7A7-Y8QZ65MW-8738SAZP} 2010-01-04 07:12:31  DTAA/RES0962  PC-3736   
1  {G7A8-G1OB94NR-3006NTXH} 2010-01-04 07:35:40  DTAA/BJC0569  PC-2588   
2  {R3L8-N0LW95FR-8358LLXS} 2010-01-04 08:00:38  DTAA/EMZ0196  PC-1479   
3  {I2F1-B5FB51FL-3128HBUL} 2010-01-04 08:02:14  DTAA/ZKH0388  PC-1021   
4  {P7R6-C5TV18CT-1677DWWM} 2010-01-04 08:20:17  DTAA/RES0962  PC-3736   

     activity  
0     Connect  
1     Connect  
2     Connect  
3     Connect  
4  Disconnect  


In [6]:
# HTTP file path
http_file = "../data/raw/r1/r1/http.csv"

# Count HTTP rows while processing in chunks
http_rows = 0
chunk_number = 0

for chunk in pd.read_csv(
    http_file,
    header=None,
    names=["id", "date", "user", "pc", "url"],
    chunksize=100000
):
    # Convert date
    chunk["date"] = pd.to_datetime(
        chunk["date"],
        format="%m/%d/%Y %H:%M:%S"
    )

    http_rows += len(chunk)
    chunk_number += 1

    print(
        f"Processed chunk {chunk_number}: "
        f"{len(chunk)} rows"
    )

print("\nTotal HTTP rows processed:", http_rows)

Processed chunk 1: 100000 rows
Processed chunk 2: 100000 rows
Processed chunk 3: 100000 rows
Processed chunk 4: 100000 rows
Processed chunk 5: 100000 rows
Processed chunk 6: 100000 rows
Processed chunk 7: 100000 rows
Processed chunk 8: 100000 rows
Processed chunk 9: 100000 rows
Processed chunk 10: 100000 rows
Processed chunk 11: 100000 rows
Processed chunk 12: 100000 rows
Processed chunk 13: 100000 rows
Processed chunk 14: 100000 rows
Processed chunk 15: 100000 rows
Processed chunk 16: 100000 rows
Processed chunk 17: 100000 rows
Processed chunk 18: 100000 rows
Processed chunk 19: 100000 rows
Processed chunk 20: 100000 rows
Processed chunk 21: 100000 rows
Processed chunk 22: 100000 rows
Processed chunk 23: 100000 rows
Processed chunk 24: 100000 rows
Processed chunk 25: 100000 rows
Processed chunk 26: 100000 rows
Processed chunk 27: 100000 rows
Processed chunk 28: 100000 rows
Processed chunk 29: 100000 rows
Processed chunk 30: 100000 rows
Processed chunk 31: 100000 rows
Processed chunk 3

In [7]:
# Create HTTP behavioral features

http_features = {}

for chunk in pd.read_csv(
    http_file,
    header=None,
    names=["id", "date", "user", "pc", "url"],
    chunksize=100000
):
    # Convert date
    chunk["date"] = pd.to_datetime(
        chunk["date"],
        format="%m/%d/%Y %H:%M:%S"
    )

    # Extract hour
    chunk["hour"] = chunk["date"].dt.hour

    # Mark after-hours activity
    chunk["after_hours"] = (
        (chunk["hour"] < 7) |
        (chunk["hour"] >= 19)
    ).astype(int)

    # Aggregate by user
    grouped = chunk.groupby("user").agg(
        http_activity_count=("url", "count"),
        unique_urls=("url", "nunique"),
        after_hours_activity=("after_hours", "sum")
    )

    # Add results from this chunk
    for user, row in grouped.iterrows():

        if user not in http_features:
            http_features[user] = {
                "http_activity_count": 0,
                "unique_urls": 0,
                "after_hours_activity": 0
            }

        http_features[user]["http_activity_count"] += row["http_activity_count"]
        http_features[user]["unique_urls"] += row["unique_urls"]
        http_features[user]["after_hours_activity"] += row["after_hours_activity"]

print("Number of users processed:", len(http_features))

Number of users processed: 1000


# Convert HTTP features dictionary into DataFrame

http_features_df = pd.DataFrame.from_dict(
    http_features,
    orient="index"
)

# Make user a normal column
http_features_df.index.name = "user"
http_features_df = http_features_df.reset_index()

print("HTTP feature shape:", http_features_df.shape)
print("\nColumns:")
print(http_features_df.columns.tolist())

print("\nFirst 5 rows:")
print(http_features_df.head())

In [8]:
# Convert HTTP features dictionary into DataFrame

http_features_df = pd.DataFrame.from_dict(
    http_features,
    orient="index"
)

# Make user a normal column
http_features_df.index.name = "user"
http_features_df = http_features_df.reset_index()

print("HTTP feature shape:", http_features_df.shape)
print("\nColumns:")
print(http_features_df.columns.tolist())

print("\nFirst 5 rows:")
print(http_features_df.head())

HTTP feature shape: (1000, 4)

Columns:
['user', 'http_activity_count', 'unique_urls', 'after_hours_activity']

First 5 rows:
           user  http_activity_count  unique_urls  after_hours_activity
0  DTAA/AAA0371                 7085         2241                     0
1  DTAA/AAC0344                14884         2973                     0
2  DTAA/AAC0599                14926         2940                     0
3  DTAA/AAH0734                16740         3216                     0
4  DTAA/AAK0658                14715         2933                     0


In [9]:
print(http_features_df.describe())

       http_activity_count  unique_urls  after_hours_activity
count          1000.000000  1000.000000           1000.000000
mean           3451.665000  1571.830000             64.683000
std            3868.476028   643.294929            205.674034
min               5.000000     5.000000              0.000000
25%            1637.000000  1218.500000              0.000000
50%            1935.500000  1491.000000              0.000000
75%            2527.000000  1827.000000             20.250000
max           20966.000000  4459.000000           2533.000000


In [10]:
# Create Logon behavioral features

logon_features = logon.groupby("user").agg(
    total_logon_events=("activity", "count"),
    unique_pcs=("pc", "nunique")
).reset_index()

# Count logons separately
logon_count = (
    logon[logon["activity"] == "Logon"]
    .groupby("user")
    .size()
    .reset_index(name="logon_count")
)

# Count logoffs separately
logoff_count = (
    logon[logon["activity"] == "Logoff"]
    .groupby("user")
    .size()
    .reset_index(name="logoff_count")
)

# After-hours logons
logon["hour"] = logon["date"].dt.hour

after_hours_logon = (
    logon[
        (logon["activity"] == "Logon") &
        ((logon["hour"] < 7) | (logon["hour"] >= 19))
    ]
    .groupby("user")
    .size()
    .reset_index(name="after_hours_logon")
)

# Merge features
logon_features = logon_features.merge(
    logon_count,
    on="user",
    how="left"
)

logon_features = logon_features.merge(
    logoff_count,
    on="user",
    how="left"
)

logon_features = logon_features.merge(
    after_hours_logon,
    on="user",
    how="left"
)

# Replace missing values with 0
logon_features = logon_features.fillna(0)

print("Logon feature shape:", logon_features.shape)
print(logon_features.head())

Logon feature shape: (1000, 6)
           user  total_logon_events  unique_pcs  logon_count  logoff_count  \
0  DTAA/AAA0371                 843          16          536           307   
1  DTAA/AAC0344                 690           1          345           345   
2  DTAA/AAC0599                 690           1          345           345   
3  DTAA/AAH0734                 690           1          345           345   
4  DTAA/AAK0658                 690           1          345           345   

   after_hours_logon  
0                0.0  
1                0.0  
2                0.0  
3                0.0  
4                0.0  


Create Device behavioral features
We'll calculate:

total device events
device connections
device disconnections
number of PCs where devices were used
after-hours device activity

In [11]:
# Create Device behavioral features

device_features = device.groupby("user").agg(
    total_device_events=("activity", "count"),
    device_pcs=("pc", "nunique")
).reset_index()

# Device connections
device_connect = (
    device[device["activity"] == "Connect"]
    .groupby("user")
    .size()
    .reset_index(name="device_connect_count")
)

# Device disconnections
device_disconnect = (
    device[device["activity"] == "Disconnect"]
    .groupby("user")
    .size()
    .reset_index(name="device_disconnect_count")
)

# After-hours device activity
device["hour"] = device["date"].dt.hour

after_hours_device = (
    device[
        (device["hour"] < 7) | (device["hour"] >= 19)
    ]
    .groupby("user")
    .size()
    .reset_index(name="after_hours_device")
)

# Merge features
device_features = device_features.merge(
    device_connect,
    on="user",
    how="left"
)

device_features = device_features.merge(
    device_disconnect,
    on="user",
    how="left"
)

device_features = device_features.merge(
    after_hours_device,
    on="user",
    how="left"
)

# Replace missing values
device_features = device_features.fillna(0)

print("Device feature shape:", device_features.shape)
print(device_features.head())

Device feature shape: (228, 6)
           user  total_device_events  device_pcs  device_connect_count  \
0  DTAA/ABB0272                  655           1                   345   
1  DTAA/ABS0726                   62           1                    31   
2  DTAA/ACH0803                  516           1                   273   
3  DTAA/AFF0760                    6           1                     3   
4  DTAA/AFH0331                  520           1                   274   

   device_disconnect_count  after_hours_device  
0                      310                 0.0  
1                       31                62.0  
2                      243                 0.0  
3                        3                 6.0  
4                      246                 7.0  


Merge the features

We will use HTTP users as the base, because HTTP + Logon cover all 1,000 users.

In [12]:
# Combine HTTP and Logon features

features = http_features_df.merge(
    logon_features,
    on="user",
    how="left"
)

# Add Device features
features = features.merge(
    device_features,
    on="user",
    how="left"
)

# Device activity is unavailable for some users,
# so replace those missing values with 0
device_columns = [
    "total_device_events",
    "device_pcs",
    "device_connect_count",
    "device_disconnect_count",
    "after_hours_device"
]

features[device_columns] = features[device_columns].fillna(0)

print("Final feature shape:", features.shape)

print("\nColumns:")
print(features.columns.tolist())

print("\nFirst 5 rows:")
print(features.head())

Final feature shape: (1000, 14)

Columns:
['user', 'http_activity_count', 'unique_urls', 'after_hours_activity', 'total_logon_events', 'unique_pcs', 'logon_count', 'logoff_count', 'after_hours_logon', 'total_device_events', 'device_pcs', 'device_connect_count', 'device_disconnect_count', 'after_hours_device']

First 5 rows:
           user  http_activity_count  unique_urls  after_hours_activity  \
0  DTAA/AAA0371                 7085         2241                     0   
1  DTAA/AAC0344                14884         2973                     0   
2  DTAA/AAC0599                14926         2940                     0   
3  DTAA/AAH0734                16740         3216                     0   
4  DTAA/AAK0658                14715         2933                     0   

   total_logon_events  unique_pcs  logon_count  logoff_count  \
0                 843          16          536           307   
1                 690           1          345           345   
2                 690          

In [13]:
# Remove the approximate unique URL feature
features = features.drop(columns=["unique_urls"])

print("Updated feature shape:", features.shape)

print("\nFeature columns:")
print(features.columns.tolist())

Updated feature shape: (1000, 13)

Feature columns:
['user', 'http_activity_count', 'after_hours_activity', 'total_logon_events', 'unique_pcs', 'logon_count', 'logoff_count', 'after_hours_logon', 'total_device_events', 'device_pcs', 'device_connect_count', 'device_disconnect_count', 'after_hours_device']


In [14]:
print("Missing values:")
print(features.isnull().sum())

print("\nDuplicate users:")
print(features["user"].duplicated().sum())

Missing values:
user                       0
http_activity_count        0
after_hours_activity       0
total_logon_events         0
unique_pcs                 0
logon_count                0
logoff_count               0
after_hours_logon          0
total_device_events        0
device_pcs                 0
device_connect_count       0
device_disconnect_count    0
after_hours_device         0
dtype: int64

Duplicate users:
0


In [15]:
# Create processed directory if it does not exist
os.makedirs("../data/processed", exist_ok=True)

# Save LeakMind feature dataset
output_file = "../data/processed/leakmind_features.csv"

features.to_csv(output_file, index=False)

print("Saved successfully!")
print("File:", output_file)
print("Shape:", features.shape)

Saved successfully!
File: ../data/processed/leakmind_features.csv
Shape: (1000, 13)
